In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval
except Exception:
    pass

In [2]:
from collections.abc import Iterator
from typing import Any, NamedTuple, cast

import numpy as np

from dataeval.config import set_max_processes
from dataeval.data import EvenlySpaced, Representative, SequenceFrames, Stride
from dataeval.protocols import DatasetMetadata, DatumMetadata, MultiobjectTrackingDatum, VideoFrame
from dataeval.types import SourceIndex

set_max_processes(1)

In [3]:
N_FRAMES, STILL = 60, 40


def scene(seed: int) -> list[np.ndarray]:
    """A still scene with sensor noise for 40 frames, then a different picture at every frame."""
    rng = np.random.default_rng(seed)
    still = rng.integers(0, 255, (3, 16, 16))
    noisy = [np.clip(still + rng.integers(-3, 4, still.shape), 0, 255) for _ in range(STILL)]
    busy = [rng.integers(0, 255, (3, 16, 16)) for _ in range(N_FRAMES - STILL)]
    return [frame.astype(np.uint8) for frame in noisy + busy]


class FrameTargets(NamedTuple):
    """The boxes in one frame, which MAITE calls a `SingleFrameObjectTrackingTarget`."""

    boxes: np.ndarray
    labels: np.ndarray
    scores: np.ndarray
    track_ids: np.ndarray


class VideoTargets(NamedTuple):
    """The boxes in every frame of one video, which MAITE calls a `MultiobjectTrackingTarget`."""

    frame_tracks: list[FrameTargets]


class Frame(NamedTuple):
    """One decoded frame, which MAITE calls a `VideoFrame`."""

    pixels: np.ndarray
    time_s: float
    pts: int
    frame_index: int


class Stream:
    """The frames of one video."""

    def __init__(self, frames: list[np.ndarray]) -> None:
        self._frames = frames

    def __iter__(self) -> Iterator[VideoFrame]:
        for index, pixels in enumerate(self._frames):
            yield Frame(pixels=pixels, time_s=index / 30, pts=index, frame_index=index)


class Videos:
    """A minimal MAITE multi-object tracking dataset."""

    def __init__(self, n_videos: int) -> None:
        self.metadata = DatasetMetadata({"id": "videos", "index2label": {0: "car"}})
        self._scenes = [scene(seed) for seed in range(n_videos)]

    def __len__(self) -> int:
        return len(self._scenes)

    def __getitem__(self, index: int) -> MultiobjectTrackingDatum:
        track = FrameTargets(
            boxes=np.array([[2, 2, 8, 8]], dtype=np.float32),
            labels=np.array([0], dtype=np.int64),
            scores=np.ones(1, dtype=np.float32),
            track_ids=np.array([0], dtype=np.int64),
        )
        target = VideoTargets(frame_tracks=[track] * N_FRAMES)
        return Stream(self._scenes[index]), target, cast(DatumMetadata, {"id": f"video_{index}"})


dataset = Videos(3)
print(f"{len(dataset)} videos of {N_FRAMES} frames")

3 videos of 60 frames


In [4]:
selections = {
    "Stride": SequenceFrames(dataset, Stride(10, jitter=True, seed=0)),
    "EvenlySpaced": SequenceFrames(dataset, EvenlySpaced(4, jitter=True, seed=0)),
    "Representative": SequenceFrames(dataset, Representative(6, batch_size=16)),
}
for name, frames in selections.items():
    first = [a.key for a in frames.source_indices if a.item == 0]
    print(f"{name:<15} {len(frames):>2} frames, video 0 keeps {first}")

Stride          18 frames, video 0 keeps [5, 10, 22, 30, 40, 56]
EvenlySpaced    12 frames, video 0 keeps [8, 15, 34, 45]
Representative  18 frames, video 0 keeps [16, 20, 25, 32, 51, 52]


In [5]:
frames = selections["Representative"]
represented = [dict(meta)["frames_represented"] for _, _, meta in frames.stream()]
sources = frames.source_indices
for address, weight in zip(sources[:6], represented[:6], strict=False):
    print(f"video {address.item}, frame {address.key:>2} stands for {weight:>4.0f} frames")

kept = frames.to_source([SourceIndex(3)])[0]
print(f"\nview position 3 is {kept!r}")

video 0, frame 16 stands for   20 frames
video 0, frame 20 stands for    3 frames
video 0, frame 25 stands for   17 frames
video 0, frame 32 stands for    7 frames
video 0, frame 51 stands for   10 frames
video 0, frame 52 stands for    3 frames

view position 3 is SourceIndex(0, 32, 'unit')


In [6]:
# TEST ASSERTION CELL ###


def weights(frames: SequenceFrames) -> list[float]:
    return [cast(dict[str, Any], meta)["frames_represented"] for _, _, meta in frames.stream()]


for name, frames in selections.items():
    for item in range(len(dataset)):
        keys = [a.key for a in frames.source_indices if a.item == item and a.key is not None]
        assert keys == sorted(keys)
        assert len(set(keys)) == len(keys)
        if name != "Stride":
            assert len(keys) == {"EvenlySpaced": 4, "Representative": 6}[name]
    for item in range(len(dataset)):
        owned = [w for a, w in zip(frames.source_indices, weights(frames), strict=True) if a.item == item]
        assert sum(owned) == N_FRAMES
again = SequenceFrames(dataset, Stride(10, jitter=True, seed=0)).source_indices
other = SequenceFrames(dataset, Stride(10, jitter=True, seed=1)).source_indices
assert again == selections["Stride"].source_indices
assert other != again
assert (
    SequenceFrames(dataset, Representative(6, batch_size=16)).source_indices
    == selections["Representative"].source_indices
)
assert kept == sources[3]